Part 1: Tensors, Autograd & The "Forward Pass"

In [1]:
#importing basic dependencies
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Configuration
BATCH_SIZE = 64
LR = 0.01

In [3]:
#Loading CIFAR-10 Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

#Normalizing pixel values from -1 to 1
x_train = (x_train.astype('float32') - 127.5) / 127.5
x_test = (x_test.astype('float32') - 127.5) / 127.5

In [4]:
#Creating tf.data.Dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE)

In [5]:
print(f"Train Data Shape:{x_train.shape}")

Train Data Shape:(50000, 32, 32, 3)


In [6]:
#Showing original shape of images along with batch size 64
batch_images = tf.random.normal((64, 32, 32, 3))
print(f"Original Shape: {batch_images.shape}")

Original Shape: (64, 32, 32, 3)


In [7]:
#Flattening image to pass through ANN
flat_images = tf.reshape(batch_images, (64, -1))
print(f"Flattened Shape: {flat_images.shape}")

Flattened Shape: (64, 3072)


In [8]:
#Performing the operation: Y=X.W+B

X = flat_images
W = tf.Variable(tf.random.normal((3072, 128)))
B = tf.Variable(tf.random.normal((128,)))
output = tf.matmul(X, W) + B

In [9]:
#tape is used to ensure that loss is remembered needed for back propogation
with tf.GradientTape() as tape:
    output = tf.matmul(X, W) + B
    loss = tf.reduce_sum(output) # returns a scalar by taking in all elements of a tensor

In [10]:
grads = tape.gradient(loss, W)

print(f"\nGradient exists for W? {grads is not None}")
print(f"Gradient shape for W: {grads.shape}")


Gradient exists for W? True
Gradient shape for W: (3072, 128)


1.Why we flatten images before feeding them into a standard Feed Forward Network?
We flatten images because an ANN can perform computations on 1D vectors. It cannot input an image with x*y*z resolutions. Therefore we flatten the image before putting it in the ANN.
2.The difference between a Tensor and a NumPy array regarding the computational graph.
A tensor participates in a computational graph that records operations for automatic differentiation, whereas a NumPy array only stores numerical values and has no knowledge of how those values were computed.

Part 2: FFNN, Loss & Gradient Descent (The Training
Loop)

In [11]:
#ANN with no dropout
class SimpleANN(tf.keras.Model):
    def __init__(self):
        super(SimpleANN, self).__init__()
        self.flatten = layers.Flatten()
        # Input: 3072 (32x32x3), Hidden 1: 256
        self.fc1 = layers.Dense(256, activation='relu')
        self.fc2 = layers.Dense(128, activation='relu')
        self.fc3 = layers.Dense(10) #Softmax applied in loss
    def call(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return self.fc3(x)

In [12]:
model_ann = SimpleANN()
loss_fn = losses.SparseCategoricalCrossentropy(from_logits=True)#from_logits here means the model is outputting logits, not probabilities

In [13]:
loss_history = []

for epoch in range(5):
    epoch_loss_sum = 0
    num_batches = 0

    for images, labels in train_ds:
        #Forward Pass & Record Operations
        with tf.GradientTape() as tape:
            predictions = model_ann(images, training=True)
            loss = loss_fn(labels, predictions)

        #Backward Pass, computing gradients
        gradients = tape.gradient(loss, model_ann.trainable_variables)

        #Updating each parameter: W =W-(LR * Gradient)
        for param, grad in zip(model_ann.trainable_variables, gradients):
            param.assign_sub(LR * grad)

        epoch_loss_sum += loss.numpy()
        num_batches += 1

    avg_loss = epoch_loss_sum / num_batches
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

Epoch 1: Loss = 1.7984
Epoch 2: Loss = 1.5810
Epoch 3: Loss = 1.4895
Epoch 4: Loss = 1.4232
Epoch 5: Loss = 1.3686


In [14]:
correct = 0
total = 0

for images, labels in test_ds:
    logits = model_ann(images, training=False)

    predictions = tf.argmax(logits, axis=1)

    labels = tf.cast(tf.squeeze(labels), tf.int64)

    correct += tf.reduce_sum(
        tf.cast(predictions == labels, tf.int32)
    ).numpy()

    total += labels.shape[0]

test_accuracy =correct/total
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.4919


A normal ANN gave an accuracy of 49.19 percent after 5 epochs

---



1.What happened to the Loss value over 5 epochs? (Did it minimize?)
It reduced considerably due to gradient descent which was converging to global minima
2.How changing the Learning Rate affects the speed of loss minimization.
w=w-alpha*(dJ/dw)
Here alpha is learning rate. So as learning rate is increased, loss minimization is faster. However having a large learning rate can cause the loss to deviate from the minima

Part 3: Regularization & Dropout

In [15]:
small_train_ds = (
    tf.data.Dataset
    .from_tensor_slices((x_train, y_train))
    .take(100)        # 100 IMAGES
    .shuffle(100)
    .batch(BATCH_SIZE)
)

val_ds = test_ds


In [16]:
model = SimpleANN()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

LR = 0.01
EPOCHS = 30

In [18]:
print("Batches:", sum(1 for _ in small_train_ds))
print("Samples:", sum(tf.size(y).numpy() for _, y in small_train_ds))


Batches: 2
Samples: 100


In [21]:
def compute_accuracy(model, dataset):
    correct = 0
    total = 0

    for images, labels in dataset:
        logits = model(images, training=False)
        preds = tf.argmax(logits, axis=1, output_type=tf.int32)
        labels = tf.cast(tf.squeeze(labels), tf.int32)

        correct += tf.reduce_sum(tf.cast(preds == labels, tf.int32)).numpy()
        total += tf.size(labels).numpy()

    return correct / total


In [22]:
train_acc_history = []
val_acc_history = []

for epoch in range(EPOCHS):

    for images, labels in small_train_ds:
        with tf.GradientTape() as tape:
            logits = model(images, training=True)
            loss = loss_fn(labels, logits)

        grads = tape.gradient(loss, model.trainable_variables)

        for param, grad in zip(model.trainable_variables, grads):
            param.assign_sub(LR * grad)

    train_acc = compute_accuracy(model, small_train_ds)
    val_acc = compute_accuracy(model, val_ds)

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch {epoch+1:02d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")


Epoch 01 | Train Acc: 0.9900 | Val Acc: 0.1949
Epoch 02 | Train Acc: 1.0000 | Val Acc: 0.1946
Epoch 03 | Train Acc: 1.0000 | Val Acc: 0.1956
Epoch 04 | Train Acc: 1.0000 | Val Acc: 0.1956
Epoch 05 | Train Acc: 1.0000 | Val Acc: 0.1950
Epoch 06 | Train Acc: 1.0000 | Val Acc: 0.1964
Epoch 07 | Train Acc: 1.0000 | Val Acc: 0.1966
Epoch 08 | Train Acc: 1.0000 | Val Acc: 0.1971
Epoch 09 | Train Acc: 1.0000 | Val Acc: 0.1964
Epoch 10 | Train Acc: 1.0000 | Val Acc: 0.1964
Epoch 11 | Train Acc: 1.0000 | Val Acc: 0.1965
Epoch 12 | Train Acc: 1.0000 | Val Acc: 0.1973
Epoch 13 | Train Acc: 1.0000 | Val Acc: 0.1981
Epoch 14 | Train Acc: 1.0000 | Val Acc: 0.1977
Epoch 15 | Train Acc: 1.0000 | Val Acc: 0.1972
Epoch 16 | Train Acc: 1.0000 | Val Acc: 0.1985
Epoch 17 | Train Acc: 1.0000 | Val Acc: 0.1983
Epoch 18 | Train Acc: 1.0000 | Val Acc: 0.2003
Epoch 19 | Train Acc: 1.0000 | Val Acc: 0.1992
Epoch 20 | Train Acc: 1.0000 | Val Acc: 0.1995
Epoch 21 | Train Acc: 1.0000 | Val Acc: 0.2000
Epoch 22 | Tr

Clearly the model was overfiiting and therefore generalizing very poorly

In [63]:
class DropoutANN(tf.keras.Model):
    def __init__(self):
        super(DropoutANN, self).__init__()
        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(256, activation='relu')
        self.dropout1 = layers.Dropout(0.2)
        self.fc2 = layers.Dense(128, activation='relu')
        self.dropout2 = layers.Dropout(0.2)
        self.fc3 = layers.Dense(10)

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout1(x, training=training)
        x = self.fc2(x)
        x = self.dropout2(x, training=training)
        return self.fc3(x)

In [64]:
def compute_accuracy(model, dataset):
    correct = 0
    total = 0
    for images, labels in dataset:
        logits = model(images, training=False)
        preds = tf.argmax(logits, axis=1, output_type=tf.int32)
        labels = tf.cast(tf.squeeze(labels), tf.int32)  # ensures dtype matches
        correct += tf.reduce_sum(tf.cast(preds == labels, tf.int32)).numpy()
        total += tf.size(labels).numpy()
    return (correct / total) * 100

In [65]:
EPOCHS = 5
LR = 0.001

model_dropout = DropoutANN()
loss_fn = losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = optimizers.Adam(learning_rate=LR)

train_acc_history = []
val_acc_history = []
loss_history = []

In [66]:
for epoch in range(EPOCHS):
    epoch_loss = tf.keras.metrics.Mean()

    for images, labels in train_ds:
        with tf.GradientTape() as tape:
            logits = model_dropout(images, training=True)
            loss = loss_fn(labels, logits)

        grads = tape.gradient(loss, model_dropout.trainable_variables)
        optimizer.apply_gradients(zip(grads, model_dropout.trainable_variables))
        epoch_loss.update_state(loss)

    # Compute accuracy on train and test sets
    train_acc = compute_accuracy(model_dropout, train_ds)
    test_acc = compute_accuracy(model_dropout, test_ds)

    train_acc_history.append(train_acc)
    val_acc_history.append(test_acc)
    loss_history.append(epoch_loss.result().numpy())

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss.result():.4f} | "
          f"Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

Epoch 01 | Loss: 1.8283 | Train Acc: 44.91% | Test Acc: 43.83%
Epoch 02 | Loss: 1.6342 | Train Acc: 48.61% | Test Acc: 46.98%
Epoch 03 | Loss: 1.5575 | Train Acc: 50.75% | Test Acc: 48.24%
Epoch 04 | Loss: 1.4996 | Train Acc: 52.50% | Test Acc: 49.41%
Epoch 05 | Loss: 1.4567 | Train Acc: 53.86% | Test Acc: 49.57%


The ANN with Dropout gave marginally better results with accuracy 49.57

1.Did Dropout slow down the training accuracy? (It usually does).

Yes, it did

2.Did Dropout improve the validation accuracy? (It should).

Yes, it did

Part 4: Fundamentals of CNN (Convolutional Neural
Networks)

In [59]:
class SimpleCNN(tf.keras.Model):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = layers.Conv2D(32, 3, activation='relu')
        self.pool1 = layers.MaxPooling2D()
        self.conv2 = layers.Conv2D(64, 3, activation='relu')
        self.pool2 = layers.MaxPooling2D()
        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(128, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.fc2 = layers.Dense(10)  # logits

    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x, training=training)
        return self.fc2(x)

In [60]:
def compute_accuracy(model, dataset):
    correct = 0
    total = 0
    for images, labels in dataset:
        logits = model(images, training=False)
        preds = tf.argmax(logits, axis=1, output_type=tf.int32)
        labels = tf.cast(tf.squeeze(labels), tf.int32)  # ensure same dtype
        correct += tf.reduce_sum(tf.cast(preds == labels, tf.int32)).numpy()
        total += tf.size(labels).numpy()
    return correct / total

In [61]:
EPOCHS = 5
LR = 0.001

model_cnn = SimpleCNN()
loss_fn = losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = optimizers.Adam(learning_rate=LR)

train_acc_history = []
val_acc_history = []

In [62]:
for epoch in range(EPOCHS):
    epoch_loss = tf.keras.metrics.Mean()

    for images, labels in train_ds:
        with tf.GradientTape() as tape:
            logits = model_cnn(images, training=True)
            loss = loss_fn(labels, logits)

        grads = tape.gradient(loss, model_cnn.trainable_variables)
        optimizer.apply_gradients(zip(grads, model_cnn.trainable_variables))
        epoch_loss.update_state(loss)

    # Compute train and validation accuracy
    train_acc = compute_accuracy(model_cnn, train_ds)
    val_acc = compute_accuracy(model_cnn, test_ds)

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss.result():.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

Epoch 01 | Loss: 1.4504 | Train Acc: 0.6189 | Val Acc: 0.6007
Epoch 02 | Loss: 1.0884 | Train Acc: 0.6963 | Val Acc: 0.6621
Epoch 03 | Loss: 0.9417 | Train Acc: 0.7414 | Val Acc: 0.6925
Epoch 04 | Loss: 0.8455 | Train Acc: 0.7667 | Val Acc: 0.7066
Epoch 05 | Loss: 0.7679 | Train Acc: 0.7966 | Val Acc: 0.7130


Clearly even a Simple CNN outperforms ANNs on a CIFAR-10 dataset with test accuracy 71.30

1.Why the CNN achieves higher accuracy with potentially fewer parameters than a massive ANN.
Because CNNs with thier onvolution operation can extract the important feautures from an image. Although information is being lost, the most important feautures are being retained in each convolution operation.
2.How the Filter/Kernel works to detect edges/shapes (The "Feature Map" concept).
By applying filters, edges can be detected.